# HiC-ECC | Enhance (pyECC)
Visualization of enhanced Hi-C matrices across tissues with CHESS unique regions.

## Config
**Edit only this cell.**

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.colors as colors
from sklearn.preprocessing import normalize
from matplotlib.patches import Arc, Rectangle
from matplotlib.colors import ListedColormap
from matplotlib.path import Path
from matplotlib.patches import PathPatch
from scipy.ndimage import label
from scipy.stats import ranksums
import pandas as pd, cooler, os
try:
    import fanc
    import fanc.plotting as fancplot
    FANC_AVAILABLE = True
except ImportError:
    FANC_AVAILABLE = False

# Region
CHR        = 4
START      = 55_000_000
END        = 56_000_000
GENE       = 'Test_Region'     # used in output filename
RESOLUTION = 10_000

# Tissues & colors (auto-assigned if COLORS = [])
TISSUES = [
    'Brain', 'Kidney', 'Large_Intestine', 'Liver',
    'Lung', 'Pancreas', 'Small_Intestine', 'Spleen',
]
COLORS = [   # leave empty [] to auto-assign
    '#D51F26', '#89288F', '#272E6A', '#F47D2B',
    '#FEE500', '#8A9FD1', '#C06CAB', '#F9B712',
]

# Paths
COOL_DIR   = '/path/to/cool_files'           # {COOL_DIR}/{tissue}/{tissue}_deephic.10kb.cool
COOL_SUFFIX = '_deephic.10kb.cool'
UNIQ_DIR   = '/path/to/chess_output/unique_regions'
GTF_PATH   = '/path/to/annotation.gtf'      # only used if SHOW_GENES = True

# TAD files expected at: {COOL_DIR}/{tissue}/findTADs/{tissue}_deephic_tads_domains.bed
# Loop files expected at: {COOL_DIR}/{tissue}/{tissue}_loops_highconf.bedpe

# Plot toggles
SHOW_TADS        = True
SHOW_LOOPS       = True
SHOW_ARCS        = True
SHOW_INSULATION  = True
SHOW_GENES       = True
SPECIFICITY_TISSUES = ['Small_Intestine', 'Liver']  # [] to disable entirely

# Matrix filtering
PERCENTILE     = 90
WIN_SZ         = '310kb'
MERGE_INTERVALS = True

## Helper functions

In [ ]:
def assign_colors(tissues, user_colors):
    if user_colors and len(user_colors) >= len(tissues):
        return user_colors[:len(tissues)]
    auto = plt.cm.tab20.colors
    return [colors.to_hex(auto[i % len(auto)]) for i in range(len(tissues))]

def get_unique_regions(path):
    if not os.path.exists(path):
        return None
    df = pd.read_csv(path, sep='\t', header=None)
    df = df[(df[0] == f'chr{CHR}') & (df[1] < END) & (df[2] > START)].copy()
    df.loc[df[1] < START, 1] = START
    df.loc[df[2] > END,   2] = END
    return df[[1, 2]].values.tolist()

def get_matrix_size():
    cool = f'{COOL_DIR}/{TISSUES[0]}/{TISSUES[0]}{COOL_SUFFIX}'
    return cooler.Cooler(cool).matrix(balance=False).fetch(f'chr{CHR}:{START}-{END+RESOLUTION}').shape[0]

def get_uniq_matrix(region_start, region_end, mat_size):
    s = int((region_start - START) / RESOLUTION)
    e = int((region_end   - START) / RESOLUTION)
    m = np.zeros((mat_size, mat_size), dtype=np.int8)
    bins = list(range(s, min(e + 1, mat_size)))
    for i in bins:
        for j in bins:
            if i < mat_size and j < mat_size:
                m[i, j] = 1
    return m

def merge_intervals(intervals):
    intervals.sort(key=lambda x: x[0])
    merged = [intervals[0]]
    for cur in intervals[1:]:
        if cur[0] <= merged[-1][1]:
            merged[-1] = [merged[-1][0], max(merged[-1][1], cur[1])]
        else:
            merged.append(cur)
    return merged

def combine_uniq_regions(uniq_reg_list, mat_size):
    if not uniq_reg_list:
        return np.zeros((mat_size, mat_size))
    intervals = merge_intervals(uniq_reg_list) if MERGE_INTERVALS else uniq_reg_list
    result = np.zeros((mat_size, mat_size))
    for r in intervals:
        result += get_uniq_matrix(r[0], r[1], mat_size)
    result[result > 0] = 1
    return result

def get_uniq_interaction(cool_path, uniq_mat):
    matrix = np.array(cooler.Cooler(cool_path).matrix(balance=False).fetch(f'chr{CHR}:{START}-{END+RESOLUTION}'))
    um = uniq_mat
    if um.shape != matrix.shape:
        nm = np.zeros(matrix.shape, dtype=um.dtype)
        nm[:um.shape[0], :um.shape[1]] = um[:matrix.shape[0], :matrix.shape[1]]
        um = nm
    return um * matrix

def get_nth_percentile(matrix, n):
    result = matrix.copy()
    result[result < np.percentile(result, n)] = 0
    return result

def calculate_insulation_score(matrix, window_size=10):
    n = matrix.shape[0]
    score = np.zeros(n)
    for i in range(n):
        us, ue = max(0, i - window_size), i
        ds, de = i, min(n, i + window_size)
        if ue > us and de > ds:
            score[i] = np.nanmean(matrix[us:ue, ds:de])
        else:
            score[i] = np.nan
    std = np.nanstd(score)
    if std > 0:
        score = (score - np.nanmean(score)) / std
    return score

def calculate_tissue_specificity(ref_tissue, tissue_raw_data):
    mat_size = tissue_raw_data[ref_tissue].shape[0]
    spec = np.zeros(mat_size)
    for b in range(mat_size):
        ref_sig = np.mean(np.concatenate([tissue_raw_data[ref_tissue][b, :], tissue_raw_data[ref_tissue][:, b]]))
        others  = [np.mean(np.concatenate([tissue_raw_data[t][b, :], tissue_raw_data[t][:, b]]))
                   for t in TISSUES if t != ref_tissue and t in tissue_raw_data]
        if not others:
            continue
        log2fc = np.log2((ref_sig + 0.01) / (np.mean(others) + 0.01))
        if len(others) >= 3:
            try:
                _, p = ranksums(others, [ref_sig] * len(others))
                sig = -np.log10(p) if 0 < p < 1 else (10 if p == 0 else 0)
                spec[b] = np.clip(log2fc * sig, -10, 10)
            except:
                spec[b] = np.clip(log2fc, -5, 5)
        else:
            spec[b] = np.clip(log2fc, -5, 5)
    return spec

def load_loops(bedpe_path):
    if not os.path.exists(bedpe_path):
        return []
    loops = []
    with open(bedpe_path) as f:
        for line in f:
            if line.startswith('#'): continue
            p = line.strip().split('\t')
            if len(p) < 8: continue
            chr1, s1, e1 = p[0], int(p[1]), int(p[2])
            chr2, s2, e2 = p[3], int(p[4]), int(p[5])
            if chr1 == f'chr{CHR}' and chr2 == f'chr{CHR}':
                a1, a2 = (s1+e1)//2, (s2+e2)//2
                if START <= a1 <= END and START <= a2 <= END:
                    loops.append({'anchor1': a1, 'anchor2': a2, 'score': float(p[6]), 'qvalue': float(p[7])})
    return loops

def filter_loops(loops, mat):
    return [l for l in loops
            if (b1 := (l['anchor1']-START)//RESOLUTION) < mat.shape[0]
            and (b2 := (l['anchor2']-START)//RESOLUTION) < mat.shape[1]
            and mat[b1, b2] > 0.01]

def lower_clip_path(size, ax):
    verts = [(0,0),(size,size),(0,size),(0,0)]
    codes = [Path.MOVETO, Path.LINETO, Path.LINETO, Path.CLOSEPOLY]
    return PathPatch(Path(verts, codes), transform=ax.transData)

def upper_clip_path(size, ax):
    verts = [(0,0),(size,0),(size,size),(0,0)]
    codes = [Path.MOVETO, Path.LINETO, Path.LINETO, Path.CLOSEPOLY]
    return PathPatch(Path(verts, codes), transform=ax.transData)

def merge_overlapping_regions(regions, max_gap=3):
    if not regions: return []
    regions = sorted(regions, key=lambda x: x['start'])
    merged = [regions[0]]
    for cur in regions[1:]:
        last = merged[-1]
        if cur['start'] <= last['end'] + max_gap:
            s, e = min(last['start'], cur['start']), max(last['end'], cur['end'])
            merged[-1] = {'start': s, 'end': e, 'size_kb': (e-s)*RESOLUTION/1000}
        else:
            merged.append(cur)
    return merged

def calculate_region_purity(start_r, end_r, tissue_matrices):
    total_pos = (end_r - start_r) ** 2
    vals = {}
    for t, m in tissue_matrices.items():
        sl = m[start_r:end_r, start_r:end_r]
        vals[t] = {'total': np.sum(sl), 'coverage': np.sum(sl != 0) / total_pos * 100}
    total_sig = sum(v['total'] for v in vals.values())
    purity = {t: {'purity': (v['total']/total_sig*100) if total_sig > 0 else 0,
                  'coverage': v['coverage']} for t, v in vals.items()}
    dom = max(purity.items(), key=lambda x: x[1]['purity'])
    return dom[0], dom[1]['purity'], dom[1]['coverage']

print('Helper functions loaded.')

## Load tissue matrices

In [ ]:
TISSUE_COLORS = assign_colors(TISSUES, COLORS)
MAT_SIZE = get_matrix_size()

tissue_matrices  = {}   # normalized, filtered
tissue_raw_data  = {}   # raw unique interaction data (for arcs + specificity)

for tis in TISSUES:
    uniq_path = f'{UNIQ_DIR}/mm10_{tis}_chr{CHR}_{WIN_SZ}_0.bed'
    uniq = get_unique_regions(uniq_path)
    if uniq is None:
        print(f'  [skip] no unique regions: {tis}')
        continue
    cool_path = f'{COOL_DIR}/{tis}/{tis}{COOL_SUFFIX}'
    uniq_mat  = combine_uniq_regions(uniq, MAT_SIZE)
    raw       = get_uniq_interaction(cool_path, uniq_mat)
    tissue_raw_data[tis] = raw.copy()
    filt = get_nth_percentile(raw.copy(), PERCENTILE)
    norm = normalize(filt, axis=1, norm='l1')
    norm = np.maximum(norm, norm.T)
    tissue_matrices[tis] = norm
    print(f'  {tis}: shape={norm.shape}, nonzero={np.count_nonzero(raw)}')

print('Done loading.')

## Plot

In [ ]:
size    = MAT_SIZE
extent  = [0, size, size, 0]
cmaps   = [ListedColormap([c]) for c in TISSUE_COLORS]

# Figure layout
fig, ax = plt.subplots(figsize=(15, 20))
ax.plot([0, size], [0, size], color='white', linewidth=3)

if SHOW_ARCS:
    axins_arc = ax.inset_axes((0, -0.6, 1, .4))
    axins_arc.plot([0])
    max_arc_height = 0

if SHOW_INSULATION:
    axins_ins = ax.inset_axes((0, -0.8, 1, .15))
    all_ins_scores = []

spec_offset = -0.05
spec_axes   = {}

# Per-tissue plotting
for i, tis in enumerate(TISSUES):
    if tis not in tissue_matrices:
        continue
    mat  = tissue_matrices[tis]
    cmap = cmaps[i]
    col  = TISSUE_COLORS[i]

    # Hi-C matrix
    ax.imshow(mat, alpha=0.5, cmap=cmap, extent=extent,
              norm=colors.LogNorm(vmin=0.01, vmax=0.1))

    # TADs (upper triangle)
    if SHOW_TADS:
        tad_path = f'{COOL_DIR}/{tis}/findTADs/{tis}_deephic_tads_domains.bed'
        if os.path.exists(tad_path):
            with open(tad_path) as f:
                for line in f:
                    p = line.strip().split()
                    if len(p) < 3: continue
                    chrom, ts, te = p[0], int(p[1]), int(p[2])
                    if chrom == f'chr{CHR}' and ts < END and te > START:
                        ps = max(0, (ts-START)//RESOLUTION)
                        pe = min((te-START)//RESOLUTION, size)
                        verts = [(ps,ps),(ps,pe),(pe,pe),(pe,ps),(ps,ps)]
                        codes = [Path.MOVETO]+[Path.LINETO]*3+[Path.CLOSEPOLY]
                        patch = PathPatch(Path(verts, codes), facecolor='none',
                                          edgecolor=col, alpha=0.7, lw=1.5, ls='--')
                        patch.set_clip_path(upper_clip_path(size, ax))
                        ax.add_patch(patch)

    # Loops (lower triangle)
    if SHOW_LOOPS:
        loop_path = f'{COOL_DIR}/{tis}/{tis}_loops_highconf.bedpe'
        loops = filter_loops(load_loops(loop_path), mat)
        for lp in loops:
            b1 = (lp['anchor1']-START)//RESOLUTION
            b2 = (lp['anchor2']-START)//RESOLUTION
            if b1 < b2: b1, b2 = b2, b1
            rect = Rectangle((b2-1, b1-1), 2, 2, fill=False,
                              edgecolor='black', lw=1.5, ls='--', alpha=0.7)
            rect.set_clip_path(lower_clip_path(size, ax))
            ax.add_patch(rect)
        if loops:
            print(f'  {tis}: {len(loops)} loops plotted')

    # Purity boxes
    threshold = np.mean(mat) + 0.3*np.std(mat)
    labeled, n_reg = label(mat > threshold)
    detected = []
    for ridx in range(1, n_reg+1):
        rows, _ = np.where(labeled == ridx)
        if len(rows) < 3: continue
        sr, er = min(rows), max(rows)
        if (er-sr)*RESOLUTION >= 30000:
            dom_t, purity, cov = calculate_region_purity(sr, er, tissue_matrices)
            if dom_t == tis and purity > 60:
                detected.append({'start': sr, 'end': er, 'size_kb': (er-sr)*RESOLUTION/1000})
    for region in merge_overlapping_regions(detected):
        dom_t, purity, cov = calculate_region_purity(region['start'], region['end'], tissue_matrices)
        rect = Rectangle((region['start']-0.5, region['start']-0.5),
                          region['end']-region['start']+1, region['end']-region['start']+1,
                          fill=False, edgecolor='black', lw=1)
        ax.add_patch(rect)
        ax.text(region['end']+1, region['start']-1,
                f"{dom_t}\nPurity: {purity:.1f}%\nCoverage: {cov:.1f}%\nSpan: {region['size_kb']:.0f}kb",
                color='black', fontsize=9, fontweight='bold', va='top', ha='left',
                bbox=dict(boxstyle='round,pad=0.5', facecolor='white', edgecolor='black', alpha=0.8))

    # Tissue-specificity track
    if SPECIFICITY_TISSUES and tis in SPECIFICITY_TISSUES and tis in tissue_raw_data:
        spec  = calculate_tissue_specificity(tis, tissue_raw_data)
        axs   = ax.inset_axes((0, spec_offset, 1, .02))
        ma    = max(abs(spec.min()), abs(spec.max()))
        vlim  = np.clip(ma, 2, 3)
        im    = axs.imshow(spec.reshape(1,-1), aspect='auto', cmap='RdBu_r',
                           interpolation='nearest', vmin=-vlim, vmax=vlim)
        axs.set_xticks([]); axs.set_yticks([])
        axs.set_ylabel(f'Spec\n{tis}', fontsize=7, rotation=90, va='bottom')
        cax = ax.inset_axes((1.01, spec_offset, 0.01, .02))
        cb  = plt.colorbar(im, cax=cax, orientation='vertical')
        cb.set_ticks([-vlim, 0, vlim])
        cb.ax.tick_params(labelsize=7)
        spec_offset -= 0.03

    # Insulation score
    if SHOW_INSULATION:
        ins = calculate_insulation_score(mat)
        axins_ins.plot(ins, color=col, label=tis, linewidth=1.5)
        all_ins_scores.extend(ins[~np.isnan(ins)])

    # Arcs
    if SHOW_ARCS and tis in tissue_raw_data:
        arc_data = normalize(np.tril(get_nth_percentile(tissue_raw_data[tis].copy(), 99)), axis=1, norm='l1')
        for r, c in zip(*np.nonzero(arc_data)):
            w = c - r
            if w > 0:
                lw = 2 * np.sqrt(arc_data[r, c])
                axins_arc.add_patch(Arc((r + w/2, 0), w, 2*w, theta1=0, theta2=180,
                                        color=cmap(256), linewidth=lw))
                if w > max_arc_height: max_arc_height = w

    ax.tick_params(left=False, right=False, labelleft=False, labelbottom=False, bottom=False)
    print(f'  {tis} plotted')

# Legend
from matplotlib.patches import Patch
ax.legend(handles=[Patch(facecolor=TISSUE_COLORS[i], label=TISSUES[i]) for i in range(len(TISSUES))],
          loc='upper right', fontsize=10)

# Insulation finalize
if SHOW_INSULATION and all_ins_scores:
    yr = max(abs(np.percentile(all_ins_scores, 5)), abs(np.percentile(all_ins_scores, 95)))
    axins_ins.set_ylim(-yr, yr)
    axins_ins.set_xlim(0, size)
    axins_ins.set_xticks([])
    axins_ins.set_ylabel('Insulation', fontsize=10)
    axins_ins.axhline(0, color='gray', ls='--', lw=1, alpha=0.5)

# Arcs finalize
if SHOW_ARCS and max_arc_height > 0:
    axins_arc.set_ylim(max_arc_height, 0)
    axins_arc.tick_params(left=False, right=False, labelleft=False, labelbottom=False, bottom=False)
    axins_arc.set_ylabel('Enhanced - Top 1%', fontsize='medium')

# Gene track
if SHOW_GENES and FANC_AVAILABLE and os.path.exists(GTF_PATH):
    axins_bed = ax.inset_axes((0, -0.14, 1, .05))
    bed = fanc.load(GTF_PATH)
    fancplot.GenePlot(bed, ax=axins_bed, n_ticks=5, group_by='gene_name',
                      squash=True, show_labels=True).plot(f'chr{CHR}:{START}-{END}')
elif SHOW_GENES and not FANC_AVAILABLE:
    print('[warn] fanc not available, skipping gene track')

# Save
plt.tight_layout(rect=[0, 0.1, 1, 0.95])
fig.subplots_adjust(hspace=0.3)

outname = f'HiC_ECC_{GENE}_chr{CHR}_{START}_{END}_p{PERCENTILE}.pdf'
fig.savefig(outname, format='pdf', bbox_inches='tight', dpi=500)
plt.close(fig)
print(f'Saved: {outname}')